<a href="https://colab.research.google.com/github/kushalpatel92/cs521/blob/main/hw3/q1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: /content/linf_base.pt
100% 11.2M/11.2M [00:00<00:00, 24.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=993fc79d-3901-495b-9d55-ccb29cc3e8b0
To: /content/cifar10_train_10k.pt
100% 30.8M/30.8M [00:00<00:00, 89.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /content/cifar10_test_1k.pt
100% 3.08M/3.08M [00:00<00:00, 170MB/s]


In [2]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [3]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [4]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

In [5]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.empty_like(x).uniform_(-eps, eps)
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad.sign()
        delta = delta.clamp(-eps, eps)
        delta = clip_box(x, delta)
    return (x + delta).detach()

In [6]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

In [23]:
def standard_loss(model, x, y):
    model.train()
    return F.cross_entropy(model(x), y)


def linf_adv_loss(model, x, y, eps=8/255, steps=10):
    model.eval()
    x_adv = pgd_linf(
        model, x, y,
        eps=eps,
        alpha=2.5 * eps / steps,
        steps=steps
    )
    model.train()
    return F.cross_entropy(model(x_adv), y)

In [24]:
TRAIN_EPS = 0.03
EPS_LIST = [0, 0.005, 0.01, 0.02, 0.03, 0.05]

std_model = adv_train(make_model(), standard_loss, epochs=5)

adv_model = adv_train(
    make_model(), linf_adv_loss,
    epochs=5, eps=TRAIN_EPS
)

# Evaluate clean and PGD robust accuracy
def test_model(model, eps, k=20, batch_size=250):
    model.eval()
    correct = 0

    for x, y in batches(x_test, y_test, batch_size, shuffle=False):
        if eps > 0:
            x = pgd_linf(
                model, x, y,
                eps=eps,
                alpha=2.5 * eps / k,
                steps=k
            )

        with torch.no_grad():
            correct += (model(x).argmax(1) == y).sum().item()

    return 100.0 * correct / len(y_test)


print("PART (a): Clean and PGD robust accuracy")
print(f"Training epsilon for adversarial model: {TRAIN_EPS}")
print(f"{'Model':<15}" + "".join(f"{e:>9.3f}" for e in EPS_LIST))

for name, model in [("Standard", std_model), ("Adversarial", adv_model)]:
    values = [test_model(model, e) for e in EPS_LIST]
    print(f"{name:<15}" + "".join(f"{v:>9.2f}" for v in values))

epoch 1: loss 1.877 (0 s)
epoch 2: loss 1.540 (1 s)
epoch 3: loss 1.371 (1 s)
epoch 4: loss 1.224 (2 s)
epoch 5: loss 1.137 (2 s)
epoch 1: loss 2.177 (3 s)
epoch 2: loss 2.086 (6 s)
epoch 3: loss 2.043 (8 s)
epoch 4: loss 2.012 (11 s)
epoch 5: loss 2.003 (14 s)
PART (a): Clean and PGD robust accuracy
Training epsilon for adversarial model: 0.03
Model              0.000    0.005    0.010    0.020    0.030    0.050
Standard           58.80    33.10    15.40     2.10     0.40     0.00
Adversarial        38.60    36.10    34.00    29.60    25.90    17.90


In [25]:
def eval_linf(model, eps, steps=20, fgsm=False, batch_size=250):
    model.eval()
    correct = 0

    for i in range(0, len(x_test), batch_size):
        x = x_test[i:i + batch_size]
        y = y_test[i:i + batch_size]

        if eps == 0:
            x_adv = x
        elif fgsm:
            x_adv = pgd_linf(
                model, x, y,
                eps=eps, alpha=eps,
                steps=1, rand_init=False
            )
        else:
            x_adv = pgd_linf(
                model, x, y,
                eps=eps,
                alpha=2.5 * eps / steps,
                steps=steps
            )

        with torch.no_grad():
            correct += (model(x_adv).argmax(1) == y).sum().item()

    return 100.0 * correct / len(y_test)


models = {
    "Standard": std_model,
    "Adversarial": adv_model
}

print("\nPART (b): PGD vs FGSM accuracy (%)")
print(f"{'Model':<15}{'Attack':<10}" +
      "".join(f"{e:>9.3f}" for e in EPS_LIST))

for name, model in models.items():
    for attack in ["PGD", "FGSM"]:
        values = [
            eval_linf(model, e, fgsm=(attack == "FGSM"))
            for e in EPS_LIST
        ]
        print(f"{name:<15}{attack:<10}" +
              "".join(f"{v:>9.2f}" for v in values))


PART (b): PGD vs FGSM accuracy (%)
Model          Attack        0.000    0.005    0.010    0.020    0.030    0.050    0.070
Standard       PGD           58.80    33.30    15.30     2.10     0.40     0.00     0.00
Standard       FGSM          58.80    36.00    21.30     7.70     3.40     0.90     0.60
Adversarial    PGD           38.60    36.10    34.00    29.50    25.80    17.90    11.20
Adversarial    FGSM          38.60    36.00    34.30    29.90    26.40    19.50    14.50
